# 03 — The Nonlinear Schrödinger Equation

**A complex-valued PDE with periodic boundary conditions.**

The 1D focusing nonlinear Schrödinger equation (NLS) on $x \in [-5, 5]$, $t \in [0, \pi/2]$:

$$i\frac{\partial h}{\partial t} + \frac{1}{2}\frac{\partial^2 h}{\partial x^2}
+ |h|^2 h = 0$$

$$h(0, x) = 2\,\mathrm{sech}(x), \qquad
h(t, -5) = h(t, 5), \qquad h_x(t, -5) = h_x(t, 5)$$

This models wave propagation in nonlinear media (optical fibres, Bose–Einstein condensates).
The $2\,\mathrm{sech}(x)$ initial condition evolves as a **breathing higher-order soliton**
whose magnitude peaks near $t = \pi/4$. It is hard for PINNs because of: complex values, the
nonlinear coupling $|h|^2 h$, and periodic BCs on both value **and** derivative.

> **Requirements** — run `uv sync --all-packages` at the repo root first, and start Jupyter
> from the project environment (`uv run jupyter lab`) so that the `pinn` library is importable.


## Recap: How the PINN Solves This

The PINN minimises a **weighted multi-term cost function**

$$
\mathcal{L}_{total} = w_{ic}\,\mathcal{L}_{ic} + w_{bc}\,\mathcal{L}_{bc} + w_{physics}\,\mathcal{L}_{physics}
$$

via the standard five-step loop: **guess** (forward pass at collocation points) →
**physics check** (exact derivatives via `torch.autograd.grad`, plugged into the PDE residual) →
**loss** (mean squared violations) → **correction** (backprop through the residual itself) →
**iterate**. See `01_harmonic_analysis.ipynb` for the full deep-dive on this mechanism.


## 1. Setup — and the Complex-Value Trick

A real-valued network cannot output $h \in \mathbb{C}$ directly. Instead the network outputs
**two channels** interpreted as real and imaginary parts:

$$\mathrm{NN}(x, t) \to (u, v), \qquad h = u + iv$$

All derivatives are taken on $u$ and $v$ separately, then recombined:
$h_t = u_t + i v_t$, $h_{xx} = u_{xx} + i v_{xx}$. The residual
$f = i h_t + \tfrac12 h_{xx} + |h|^2 h$ is complex; the loss is $\mathrm{mean}(|f|^2)$.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.autograd as autograd
import matplotlib.pyplot as plt

from pinn import PINN, PINNTrainer, plot_contour, set_seed, setup_logging

setup_logging()  # loguru console sink (tqdm-safe)
set_seed(42)     # random / numpy / torch (CPU + CUDA)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Problem Configuration

> **Note:** the 4×100 network with 5000 collocation points for 25k epochs is a long CPU run.
> Drop `EPOCHS` to ~5000 and `HIDDEN_NEURONS` to 50 for a smoke test.


In [ ]:
# --- Physical setup ---
X_DOMAIN = (-5.0, 5.0)
T_DOMAIN = (0.0, np.pi / 2)

# --- Network / training hyperparameters ---
HIDDEN_LAYERS = 4
HIDDEN_NEURONS = 100
EPOCHS = 25_000
LR = 5e-4
N_IC, N_BC, N_PHYSICS = 100, 50, 5000
LOSS_WEIGHTS = {"ic": 1.0, "bc": 1.0, "physics": 1.0}

print(f"NLS on x in {X_DOMAIN}, t in (0, pi/2)")

## 3. Model: `ComplexPINN`

A thin wrapper around the library backbone with `output_dim=2`, splitting the outputs into
$(\mathrm{Re}\,h, \mathrm{Im}\,h)$.


In [ ]:
class ComplexPINN(nn.Module):
    """Two-output PINN returning (Re h, Im h)."""

    def __init__(self, input_dim, hidden_layers, hidden_neurons):
        super().__init__()
        self.network = PINN(input_dim, hidden_layers, hidden_neurons, output_dim=2)

    def forward(self, x, t):
        out = self.network(torch.cat([x, t], dim=1))
        return out[:, 0:1], out[:, 1:2]


model = ComplexPINN(input_dim=2, hidden_layers=HIDDEN_LAYERS, hidden_neurons=HIDDEN_NEURONS)
n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params}")

## 4. Collocation Points and Loss Terms

| Term | Enforces | Points |
|------|----------|--------|
| `ic` | $u(0,x) = 2\,\mathrm{sech}(x)$, $v(0,x) = 0$ | 100 uniform in $x$ |
| `bc` | $h(t,-5)=h(t,5)$ **and** $h_x(t,-5)=h_x(t,5)$ | 50 uniform in $t$ |
| `physics` | $\mathrm{mean}\,|i h_t + \tfrac12 h_{xx} + |h|^2 h|^2$ | 5000 uniform-random interior |

The periodic BC needs `autograd.grad` **at the boundaries** to match derivatives — the boundary
$x$-tensors must therefore carry `requires_grad=True`.


In [ ]:
x_ic = torch.linspace(*X_DOMAIN, N_IC).view(-1, 1).to(device).requires_grad_(True)
t_bc = torch.linspace(*T_DOMAIN, N_BC).view(-1, 1).to(device)

x_physics = (torch.rand(N_PHYSICS, 1) * (X_DOMAIN[1] - X_DOMAIN[0]) + X_DOMAIN[0])
t_physics = (torch.rand(N_PHYSICS, 1) * (T_DOMAIN[1] - T_DOMAIN[0]) + T_DOMAIN[0])
x_physics = x_physics.to(device).requires_grad_(True)
t_physics = t_physics.to(device).requires_grad_(True)


def pde_residual(model, x, t):
    u, v = model(x, t)
    h = u + 1j * v
    h_conj = u - 1j * v

    u_t = autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    v_t = autograd.grad(v, t, torch.ones_like(v), create_graph=True)[0]
    u_x = autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    v_x = autograd.grad(v, x, torch.ones_like(v), create_graph=True)[0]
    u_xx = autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    v_xx = autograd.grad(v_x, x, torch.ones_like(v_x), create_graph=True)[0]

    h_t = u_t + 1j * v_t
    h_xx = u_xx + 1j * v_xx

    f = 1j * h_t + 0.5 * h_xx + (h * h_conj) * h
    return torch.mean(torch.abs(f) ** 2)


def ic_loss(model):
    u, v = model(x_ic, torch.zeros_like(x_ic))
    h_exact = 2 / torch.cosh(x_ic)          # h(0,x) = 2*sech(x), purely real
    return torch.mean((u - h_exact) ** 2 + v**2)


def bc_loss(model):
    x_l = -5 * torch.ones_like(t_bc).requires_grad_(True)
    x_r = 5 * torch.ones_like(t_bc).requires_grad_(True)

    u_l, v_l = model(x_l, t_bc)
    u_r, v_r = model(x_r, t_bc)

    u_l_x = autograd.grad(u_l, x_l, torch.ones_like(u_l), create_graph=True)[0]
    v_l_x = autograd.grad(v_l, x_l, torch.ones_like(v_l), create_graph=True)[0]
    u_r_x = autograd.grad(u_r, x_r, torch.ones_like(u_r), create_graph=True)[0]
    v_r_x = autograd.grad(v_r, x_r, torch.ones_like(v_r), create_graph=True)[0]

    loss_value = torch.mean((u_l - u_r) ** 2 + (v_l - v_r) ** 2)
    loss_deriv = torch.mean((u_l_x - u_r_x) ** 2 + (v_l_x - v_r_x) ** 2)
    return loss_value + loss_deriv


def physics_loss(model):
    return pde_residual(model, x_physics, t_physics)

## 5. Training

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
trainer = PINNTrainer(model, device=device)

trainer.train(
    n_epochs=EPOCHS,
    optimizer=optimizer,
    loss_functions={"ic": ic_loss, "bc": bc_loss, "physics": physics_loss},
    weights=LOSS_WEIGHTS,
)

In [ ]:
trainer.plot_loss_history(show_total=True)

## 6. Results: Wavefunction Magnitude $|h(t,x)|$

Expect the soliton to **breathe**: the $2\,\mathrm{sech}(x)$ hump focuses to a sharp peak near
$t \approx \pi/4$, then relaxes. Boundary seams at $x = \pm 5$ mean the periodic BC has not
converged.


In [ ]:
n_x, n_t = 200, 100
x_test = torch.linspace(*X_DOMAIN, n_x).view(-1, 1).to(device)
t_test = torch.linspace(*T_DOMAIN, n_t).view(-1, 1).to(device)
X, T = torch.meshgrid(x_test.squeeze(), t_test.squeeze(), indexing="ij")

with torch.no_grad():
    u_pred, v_pred = model(X.flatten().unsqueeze(1), T.flatten().unsqueeze(1))
    h_mag = torch.sqrt(u_pred**2 + v_pred**2).cpu().numpy().reshape(n_x, n_t)

plot_contour(
    T.cpu().numpy(), X.cpu().numpy(), h_mag,
    title="PINN Solution Magnitude |h(t,x)| — Nonlinear Schrödinger",
    xlabel="t", ylabel="x", clabel="|h(t,x)|",
)

## 7. Validation Snapshots

- **$t = 0$:** $|h|$ must match the exact IC $2\,\mathrm{sech}(x)$.
- **$t = \pi/4$:** the breathing peak — amplitude should exceed the initial maximum of 2.


In [ ]:
with torch.no_grad():
    u0, v0 = model(x_test, torch.zeros_like(x_test))
    h_mag_0 = torch.sqrt(u0**2 + v0**2).cpu().numpy()

    t_peak = (np.pi / 4) * torch.ones_like(x_test)
    up, vp = model(x_test, t_peak)
    h_mag_peak = torch.sqrt(up**2 + vp**2).cpu().numpy()

x_np = x_test.cpu().numpy()
h_exact_0 = (2 / np.cosh(x_np))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(x_np, h_exact_0, "k-", label="Exact (t=0)", linewidth=2)
axes[0].plot(x_np, h_mag_0, "r--", label="PINN (t=0)", linewidth=2)
axes[0].set(title="Initial condition check", xlabel="x", ylabel="|h(0,x)|")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x_np, h_mag_peak, "b-", label="PINN (t=pi/4)", linewidth=2)
axes[1].set(title="Breathing peak near t = pi/4", xlabel="x", ylabel="|h(pi/4,x)|")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

rel_l2_ic = np.linalg.norm(h_mag_0 - h_exact_0) / np.linalg.norm(h_exact_0)
print(f"Relative L2 error at t=0 : {rel_l2_ic:.4e}")
print(f"Peak |h| at t=pi/4       : {h_mag_peak.max():.3f}   (initial max: 2.0)")
print(f"Final total loss         : {trainer.loss_history[-1]['total']:.4e}")

## 8. Takeaways

1. **Complex PDEs reduce to real ones.** Two output channels + recombination handles
   $h \in \mathbb{C}$ with zero framework changes — `PINNTrainer` never knows the field
   is complex.
2. **Periodic BCs need derivative matching.** Matching values alone leaves a kink at the seam;
   the $h_x$ constraint is what makes the boundary truly periodic.
3. **Same recipe, third equation.** Across all three notebooks the *only* things that changed
   are the residual, the conditions, and the collocation points. The cost function structure and
   the five-step loop are identical — that is the whole point of the PINN framework.

The equivalent CLI run is `uv run train-schrodinger train` (see `experiments/schrodinger/`).
